# Code as Policies with CaP-X

[CaP-X](https://github.com/capgym/cap-x) (*Code-as-Policies eXtended*) is a framework for benchmarking and improving coding agents for robot manipulation. Instead of asking a model what the arm should do next, it asks the model to **write a Python program** that solves the task by composing perception and control primitives, runs that program in simulation, and scores whether the task actually got done.

The robot is a Franka Panda again, this time in a Robosuite/MuJoCo simulation. The model that writes the program is served locally, so the reasoning stays on your Radeon GPU.

## Goals

* Understand how a code-generating agent controls a robot, and how that differs from the tool-calling agent in the previous notebook
* Configure CaP-X against your own OpenAI-compatible model server
* Read the program the model writes, and the prompt that produced it
* Measure a model by its success rate over repeated trials instead of a single demo

## Two ways to put an LLM on a robot

Notebook 03 and this notebook attempt to solve the same problem of robot manipulation through using natural language as an input, and planned motion of the arm as an output. Their approach differs with the local model's responsibility: **calling tools vs generating code.**

| | Notebook 03 - RAI | This notebook - CaP-X |
| --- | --- | --- |
| What the model emits | one tool call at a time | one Python program, up front |
| Where the model sits | inside the control loop | upstream of it |
| Model calls per task | one per step, dozens of them | one |
| What drives the robot | a LangGraph loop | CPython executing the generated code |
| Perception | a ROS 2 service, invoked as a tool | SAM3 and Contact-GraspNet, called as functions from inside the program |
| How you judge it | watch the arm | reward over N seeded trials, i.e. a success rate |
| A failure looks like | a tool call you can read | a traceback, or a program that runs cleanly and still misses |

The main tradeoff is: adaptability against efficiency. RAI can attempt any task with no preparation, but the model has to reason from scratch every time, so the hundredth run of a task costs the same as the first. CaP-X generates a program per task, and that program can then be re-run without a model at all; yet a wrong read of the scene spoils the whole episode, and every new task needs another program.

## Serve the model locally

CaP-X is compatible with plain OpenAI chat completions, therefore any server that can communicate in the protocol will do. We will use the same Lemonade server as notebooks 02 and 03, on the same port.

`ensure_lemonade` will starts `lemond` if it is not already up and loads the model onto the iGPU, hence whether you left the server running after the earlier notebooks won't affect the result for this notebook.

In [1]:
import sys

sys.path.insert(0, "/ryzers")

from capx_demo import benchmark, ensure_lemonade, show_video

MODEL = "Gemma-4-E2B-it-GGUF"
ensure_lemonade(MODEL)

Gemma-4-E2B-it-GGUF is loaded and serving on port 13305


## Start the perception services and configure CaP-X

Three servers sit behind the primitives the generated program may call. SAM3 grounds a noun phrase such as `"red cube"` into the pixels that are the cube, Contact-GraspNet turns that mask plus depth into ranked 6-DoF grasp poses, and PyRoKi solves the inverse kinematics. `FrankaControlApi` wraps them into the five functions the model is allowed to use.

The cell below is what `capx/envs/launch.py` does before its first trial, one call at a time. `LaunchArgs` is the dataclass the CLI parses its flags into, so filling one in by hand configures the framework exactly as a terminal run would, and `model` and `server_url` are the whole coupling between CaP-X and whatever is doing the reasoning. Point them at any OpenAI-compatible chat-completions endpoint and nothing else in this notebook changes.

In [2]:
from capx.envs.configs.instantiate import instantiate
from capx.envs.launch import LaunchArgs
from capx.envs.runner import _start_api_servers
from capx.utils.launch_utils import _load_config

SERVER_URL = "http://localhost:13305/api/v1/chat/completions"
CONFIG_PATH = "env_configs/cube_stack/franka_robosuite_cube_stack.yaml"

TEMPERATURE = 0.2
MAX_TOKENS = 16384

args = LaunchArgs(
    config_path=CONFIG_PATH,
    model=MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

# The config declares the servers it needs under `api_servers:`, and
# _load_config hands them back next to the factory for the env they serve.
env_factory, config, api_servers = _load_config(args)

# SAM3, Contact-GraspNet and PyRoKi, one process each.
servers = _start_api_servers(api_servers, 900.0)

# instantiate walks the _target_ entries in the YAML and builds the object graph:
# the code-execution env, the Robosuite task under it, and the FrankaControlApi
# holding the clients for the servers above.
env = instantiate(env_factory)
api = next(iter(env._apis.values()))

# reset re-samples where the cubes land on the table
obs, _ = env.reset(options={"trial": 0}, seed=0)

print("ready:", type(env).__name__, "|", type(api).__name__)

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /ryzers/cap-x/capx/third_party/robosuite/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:32)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:42)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/opt/capx-venv/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


R1Pro not installed, skipping R1Pro APIs
LIBERO not installed!
R1Pro not installed!
API server {'_target_': 'capx.serving.launch_sam3_server.main', 'device': 'cuda', 'port': 8114, 'host': '127.0.0.1'} started
API server {'_target_': 'capx.serving.launch_contact_graspnet_server.main', 'port': 8115, 'host': '127.0.0.1'} started
API server {'_target_': 'capx.serving.launch_pyroki_server.main', 'port': 8116, 'host': '127.0.0.1', 'robot': 'panda_description', 'target_link': 'panda_hand'} started


[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /ryzers/cap-x/capx/third_party/robosuite/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /ryzers/cap-x/capx/third_party/robosuite/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /ryzers/cap-x/capx/third_party/robosuite/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (ht

...

model func:  <module 'contact_graspnet_pytorch.contact_graspnet' from '/ryzers/cap-x/capx/third_party/contact_graspnet_pytorch/contact_graspnet_pytorch/contact_graspnet.py'>
/ryzers/cap-x/capx/third_party/contact_graspnet_pytorch/checkpoints/contact_graspnet/checkpoints/model.pt
=> Loading checkpoint from local file...


 40%|████      | 306.0/757.0 [00:09<00:14, 31.01it/s]INFO:     Started server process [603]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8114 (Press CTRL+C to quit)
 40%|████      | 306.0/757.0 [00:10<00:15, 29.36it/s]

API server on 127.0.0.1:8114 is ready
API server on 127.0.0.1:8115 is ready


100%|██████████| 118.0/118.0 [00:18<00:00,  6.31it/s]
INFO:jax._src.xla_bridge:Unable to initialize backend 'cuda': 
INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-08-17 22:02:14.456 | INFO     | pyroki.collision._robot_collision:from_sphere_decomposition:187 - Created RobotCollision (sphere mode) with 13 links, 41 total spheres, and 626 active pairs.
INFO:pyroki_server:PyRoki loaded and ready!
INFO:     Started server process [605]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8116 (Press CTRL+C to quit)
[robosuite INFO] Loading controller configuration from: capx/integrations/robosuite/controllers/config/robots/panda_joint_ctrl.json (composite_controlle

API server on 127.0.0.1:8116 is ready
init franka control api
init grasp net plan fn
init sam3 seg fn
ready: FrankaPickPlaceCodeEnv | FrankaControlApi


## Generate a program for the task

`ModelQueryArgs` carries the same values as `LaunchArgs`, and `query_model` is the chat-completions call underneath, so this is one HTTP round trip to the server you configured above. `obs["full_prompt"]` is the two-message conversation the environment built for the scene it reset to.

In [3]:
from capx.llm.client import ModelQueryArgs, query_model
from capx.utils.launch_utils import _extract_code

query_args = ModelQueryArgs(
    model=MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

response = query_model(query_args, obs["full_prompt"])

blocks = _extract_code(response["content"])
assert blocks, f"no code in the reply - raise MAX_TOKENS?\n{response['content'][-500:]}"

program = blocks[0]
print(program)

Time taken to query model: 15.58 seconds
import numpy as np

# --- Step 1: Get necessary poses ---

# Get pose for the green cube to calculate stacking height
green_pos, green_quat, green_extent = get_object_pose('green cube', return_bbox_extent=True)

# Sample a grasp pose for the red cube (we will use this orientation)
red_grasp_pos, red_grasp_quat = sample_grasp_pose('red cube')

# Get extent for the red cube to calculate stacking height
red_pos, red_quat, red_extent = get_object_pose('red cube', return_bbox_extent=True)

# --- Step 2: Grasp the red cube ---

# 1. Move to the grasp position with approach
goto_pose(red_grasp_pos, red_grasp_quat, z_approach=0.1)

# 2. Close the gripper
close_gripper()

# 3. Lift the cube to a safe height (at least +0.2m in Z)
lift_z = 0.2
lifted_pos = np.array([red_grasp_pos[0], red_grasp_pos[1], red_grasp_pos[2] + lift_z])
goto_pose(lifted_pos, red_grasp_quat) # Maintain the grasp orientation while lifting

# --- Step 3: Calculate placement pose ---


## Run program

**The generated code is the policy.**

If it fails, the output will display how. `No detections` means the noun phrase did not ground, a traceback means the model got Python wrong (the most common failure for small models), and a clean run below `1.0` means the geometry was wrong.

In [4]:
env.enable_video_capture(True, clear=True)

_, reward, terminated, _, info = env.step(program)

print(f"reward {reward:.3f} | solved {info['task_completed']} | terminated {terminated}")
if info["stdout"].strip():
    print("\n----- program stdout (tail) -----\n" + info["stdout"][-2000:])
if info["sandbox_rc"]:
    print("\n----- traceback (tail) -----\n" + info["stderr"][-2000:])

show_video(env)

MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/usr/local/lib/python3.12/dist-packages/_rocm_sdk_libraries_gfx1151/share/miopen/db/gfx1151_20.HIP.fdb.txt"
/ryzers/cap-x/capx/third_party/sam3/sam3/model/vitdet.py:504: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:309.)
  x = F.scaled_dot_product_attention(q, k, v)
/ryzers/cap-x/capx/third_party/sam3/sam3/model/vitdet.py:504: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:360.)
  x = F.scaled_dot_product_attention(q, k, v)


a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_npe

MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/usr/local/lib/python3.12/dist-packages/_rocm_sdk_libraries_gfx1151/share/miopen/db/gfx1151_20.HIP.fdb.txt"


Generated 2 grasps for object 1
INFO:     127.0.0.1:50466 - "POST /plan HTTP/1.1" 200 OK
Grasp sample position for red cube: [ 0.48996223  0.10227364 -0.09508278]
Grasp sample quaternion wxyz for red cube: [-0.17585823  0.22486646  0.95650228  0.06010337]
INFO:     127.0.0.1:60844 - "POST /segment HTTP/1.1" 200 OK
Object position for red cube: [ 0.49833099  0.0874119  -0.08493799]
Object quaternion wxyz for red cube: [-0.46784045  0.07469621  0.87915811  0.05125232]
Object extent for red cube: [0.05584644 0.04117704 0.03306428]


2026-08-17 22:04:16.784 | INFO     | jaxls._problem:analyze:201 - Building optimization problem with 2 terms and 1 variables: 1 costs, 0 eq_zero, 1 leq_zero, 0 geq_zero
2026-08-17 22:04:16.799 | INFO     | jaxls._problem:analyze:320 - Vectorizing constraint group with 1 constraints (constraint_leq_zero), 1 variables each: augmented_limit_residual
2026-08-17 22:04:17.140 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: _pose_cost_analytical_jac


INFO:     127.0.0.1:42810 - "POST /ik HTTP/1.1" 200 OK


2026-08-17 22:04:18.784 | INFO     | jaxls._problem:analyze:201 - Building optimization problem with 3 terms and 1 variables: 3 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
2026-08-17 22:04:18.803 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: _pose_cost_analytical_jac
2026-08-17 22:04:18.809 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: limit_residual
2026-08-17 22:04:18.815 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: limit_velocity_cost


INFO:     127.0.0.1:42824 - "POST /ik HTTP/1.1" 200 OK
INFO:     127.0.0.1:42832 - "POST /ik HTTP/1.1" 200 OK
INFO:     127.0.0.1:42836 - "POST /ik HTTP/1.1" 200 OK
INFO:     127.0.0.1:42852 - "POST /ik HTTP/1.1" 200 OK
reward 1.000 | solved True | terminated True

----- program stdout (tail) -----
Object position for green cube: [ 0.59241932  0.09679693 -0.08111711]
Object quaternion wxyz for green cube: [ 0.37774157  0.42647165 -0.73584845 -0.36600588]
Object extent for green cube: [0.07756438 0.07275727 0.06082484]
Grasp sample position for red cube: [ 0.48996223  0.10227364 -0.09508278]
Grasp sample quaternion wxyz for red cube: [-0.17585823  0.22486646  0.95650228  0.06010337]
Object position for red cube: [ 0.49833099  0.0874119  -0.08493799]
Object quaternion wxyz for red cube: [-0.46784045  0.07469621  0.87915811  0.05125232]
Object extent for red cube: [0.05584644 0.04117704 0.03306428]

Saved interaction video to /tmp/capx_notebook/video_notebook_run.mp4 (34 frames)


'/tmp/capx_notebook/video_notebook_run.mp4'

## Results over many layouts

A single trial does not measure much. Each trial reseeds the cube positions, so a program that succeeds on one arrangement can fail on the next, and only a success rate over several trials is comparable between models.

The cell below runs `launch.py` five times and prints the reward for each trial, followed by the success rate and the mean reward. Each trial also writes its own directory holding the generated program, the model's raw response, the prompt and an MP4 of the episode.

Pass `oracle=True` to run the reference program instead of calling the model.

In [5]:
trials = benchmark(
    model=MODEL,
    server_url=SERVER_URL,
    config_path=CONFIG_PATH,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    trials=5,
)

/opt/capx-venv/bin/python capx/envs/launch.py --config-path env_configs/cube_stack/franka_robosuite_cube_stack.yaml --model Gemma-4-E2B-it-GGUF --server-url http://localhost:13305/api/v1/chat/completions --temperature 0.2 --max-tokens 16384 --total-trials 5 --num-workers 1 --output-dir /tmp/capx_notebook/eval

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /ryzers/cap-x/capx/third_party/robosuite/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:32)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use

## Benchmark Episodes Visualization

In [9]:
import base64
import subprocess
from pathlib import Path

from IPython.display import HTML, display

WIDTH = 240
THUMBS = Path("/tmp/capx_notebook/thumbs")
THUMBS.mkdir(parents=True, exist_ok=True)

try:
    import imageio_ffmpeg

    FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    FFMPEG = "ffmpeg"


def scaled(video, dest, width=WIDTH):
    """Shrink a clip for embedding. -2 keeps the aspect ratio with the even
    height h264 needs; a missing ffmpeg raises, so both failures fall back."""
    try:
        done = subprocess.run(
            [FFMPEG, "-y", "-loglevel", "error", "-i", str(video),
             "-vf", f"scale={width}:-2", "-an", str(dest)],
            capture_output=True, text=True,
        )
    except OSError:
        return video
    return dest if done.returncode == 0 and dest.exists() else video


def row(items):
    return (
        '<div style="display:flex;flex-wrap:wrap;gap:12px;justify-content:center;'
        'align-items:flex-start;margin-bottom:12px">' + "".join(items) + "</div>"
    )


figures, embedded = [], 0
for t in trials:
    video = next(iter(sorted(t["dir"].glob("video_combined*.mp4"))), None)
    caption = f"trial {t['trial']} · reward {t['reward']:.3f}"
    caption += " · solved" if t["solved"] else ""
    style = 'style="margin:0;text-align:center;font:12px/1.4 sans-serif"'

    if video is None:
        figures.append(
            f'<figure {style}><div style="width:{WIDTH}px;height:{WIDTH * 3 // 4}px;'
            'display:flex;align-items:center;justify-content:center;'
            'border:1px dashed currentColor;opacity:.5">no video</div>'
            f"<figcaption>{caption}</figcaption></figure>"
        )
        continue

    source = scaled(video, THUMBS / f"trial_{t['trial']}.mp4")
    data = base64.b64encode(source.read_bytes()).decode()
    embedded += len(data)
    figures.append(
        f'<figure {style}><video src="data:video/mp4;base64,{data}" width="{WIDTH}" '
        f'controls loop muted playsinline></video>'
        f"<figcaption>{caption}</figcaption></figure>"
    )

# Half on top, the rest centred underneath: 3 over 2 for five trials
top = (len(figures) + 1) // 2
display(HTML(row(figures[:top]) + (row(figures[top:]) if len(figures) > top else "")))
print(f"{len(figures)} episodes, {embedded / 1e6:.1f} MB embedded")


5 episodes, 0.2 MB embedded


## Where the motion happens

`goto_pose` is the one primitive that moves the arm, and it does three things worth knowing about. Every pose in this API is a gripper-tip pose, so it adds the 10.7 cm offset to the `panda_hand` link before solving, or the arm drives into the table. A non-zero `z_approach` makes it solve and execute *twice*, once at a standoff along the gripper's local -Z and once at the target, which is why the task prompt insists on `z_approach=0.1`. And it hands the previous joint configuration back to the solver, which then penalises joint velocity, so the elbow does not flip across the table between waypoints.

The solve is nonlinear least squares over the joint vector, so it always returns something: an unreachable target comes back as the closest configuration it could find, with no error and no flag. When a program fails for no visible reason, suspect that first.

In [6]:
import inspect

print(inspect.getsource(type(api).goto_pose))

    def goto_pose(
        self, position: np.ndarray, quaternion_wxyz: np.ndarray, z_approach: float = 0.0
    ) -> None:
        """Go to pose using Inverse Kinematics.
        There is no need to call a second goto_pose with the same position and quaternion_wxyz after calling it with z_approach.
        Args:
            position: (3,) XYZ in meters.
            quaternion_wxyz: (4,) WXYZ unit quaternion.
            z_approach: (float) Z-axis distance offset for goto_pose insertion approach motion. Will first arrive at position + z_approach meters in Z-axis before moving to the requested pose. Useful for more precise grasp approaches. Default is 0.0.
        Returns:
            None
        """
        pos_str = np.array2string(np.asarray(position), precision=4)
        approach_info = f" (z_approach={z_approach:.3f})" if z_approach != 0.0 else ""
        self._log_step("goto_pose", f"Moving to position {pos_str}{approach_info} …")

        pos = np.asarray(position, dtype=np.floa

## System Prompt

Two messages are used as prompts for the api calls, the ones handed to `query_model` above. The system message is one sentence, and the user message is the task description followed by an `APIs:` section that is not written by hand anywhere: CaP-X renders it from the signature and docstring of every primitive the program may call. So the docstrings *are* the prompt, and it cannot drift out of date with the code.

In [7]:
system, user = obs["full_prompt"]

print(system["content"])
print(user["content"][0]["text"])

You are a helpful assistant that generates Python code to directly solve the task.

You are controlling a Franka Emika robot with the API described below.
Goal: Pick up the red cube and gently stack it on top of the green cube, then release it.

Key rules:
- The extent from get_object_pose(..., return_bbox_extent=True) is the FULL side length. Use extent[2]/2 for half-height.
- For placement orientation, reuse the grasp quaternion from sample_grasp_pose. Do NOT use the quaternion from get_object_pose (it is unreliable for orientation).
- Always use z_approach=0.1 when approaching an object for grasping or placing.
- After grasping, lift the cube to a safe height (at least +0.2m in Z) before moving laterally to the placement location.
- The stacking height formula is: place_z = green_center_z + green_extent[2]/2 + red_extent[2]/2
- Nothing should be dropped from a height. Always approach with z_approach for controlled descent.

Write ONLY executable Python code (no code fences). Import 

> **Note:** nothing in that prompt describes the scene. There is no image, no object coordinates and no joint state, so the model writes the program blind, and every fact about the actual arrangement is obtained at runtime by the program itself, through the perception servers.

## Key Takeaways

Now you know:
- How a Code-as-Policies agent differs from a tool-calling agent: one program written up front instead of one decision per step, and a success rate instead of a demo
- How to configure CaP-X against any OpenAI-compatible server, with `LaunchArgs` and `ModelQueryArgs` carrying the same `--model` and `--server-url` the CLI takes
- How the generated program gets from a noun phrase to motion: grounding into a mask, mask plus depth into a grasp pose, pose into joint angles through IK
- That the prompt is generated from the primitives' docstrings, so the API and its documentation cannot drift apart
- How to benchmark a locally served model and read the result

## What to Try Next

- Point `SERVER_URL` at a llama.cpp, Ollama or vLLM server, or at the OpenRouter proxy, and rerun from that cell down to understand how different models perform and how to use different backends
- Run `benchmark(..., oracle=True)` to see the stack succeed with no model in the loop, and use it as your control whenever a result looks wrong
- Edit a docstring in `FrankaControlApi` and watch the generated program change; you are editing the prompt. `/ryzers/cap-x` is an editable install, so the change lands on the next kernel restart
- Raise `temperature` and generate a few programs for the same scene, to see how much of the policy is the model guessing

## References

* [CaP-X](https://github.com/capgym/cap-x)
* [SAM 3](https://huggingface.co/facebook/sam3) · [Contact-GraspNet](https://github.com/NVlabs/contact_graspnet) · [PyRoKi](https://github.com/chungmin99/pyroki)
* [Robosuite](https://github.com/ARISE-Initiative/robosuite)
* [Lemonade](https://lemonade-server.ai/)